In [3]:
# AgentOps_Practical_Implementation.ipynb
#
# This notebook implements a CrewAI-based stock analysis workflow with AgentOps
# observability. The flow follows the project description by configuring API keys,
# initializing AgentOps, defining custom tools, creating specialized agents and tasks,
# assembling a crew, and executing the workflow for stock analysis.

import os
from getpass import getpass

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import yfinance as yf
from duckduckgo_search import DDGS
import agentops

# Enforce UTF-8 output so notebook logs render correctly for different character sets.
os.environ["PYTHONIOENCODING"] = "utf-8"

# Load environment variables from the local .env file if present.
load_dotenv()

# ---- API key configuration ----
# The project description expects secure data handling through local environment variables.
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

agentops_api_key = os.getenv("AGENTOPS_API_KEY")
if not agentops_api_key:
    agentops_api_key = getpass("Enter your AgentOps API key: ")
os.environ["AGENTOPS_API_KEY"] = agentops_api_key

# ---- AgentOps observability initialization ----
# Initialize AgentOps so that agent activity, tool usage, and execution traces are captured.
agentops.init(api_key=os.environ["AGENTOPS_API_KEY"])

# ---- LLM configuration ----
# Use the Gemini model for agent reasoning while keeping the execution temperature balanced.
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=api_key,
    temperature=0.7,
)


class StockSearchTool(BaseTool):
    """
    A tool to search for the latest news and updates about a stock using DuckDuckGo.
    Inherits from CrewAI's BaseTool to integrate with agent workflows.
    """
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        """
        Executes the stock news search.
        Args:
            query (str): The search query, typically a stock ticker or company name.
        Returns:
            str: A string containing summarized news highlights or an error message.
        """
        # Use DuckDuckGo to gather a compact set of recent news results for the requested stock.
        try:
            with DDGS() as ddgs:
                # Retrieve a maximum of 2 results to keep the output concise.
                results = ddgs.text(query, max_results=2)
            if not results:
                return "No news results were found."
            # Join the 'body' of the search results into a single string.
            return "\n".join([r.get("body", "") for r in results if r.get("body")])
        except Exception as exc:
            return f"News search failed: {exc}"


class YahooFinanceTool(BaseTool):
    """
    A tool to fetch the latest 1-month stock price history for a given ticker using yFinance.
    Inherits from CrewAI's BaseTool for seamless integration into agent tasks.
    """
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        """
        Fetches stock price history for a given ticker.
        Args:
            ticker (str): The stock ticker symbol (e.g., 'AAPL').
        Returns:
            str: A string representation of the last 3 days of stock price history or an error message.
        """
        # Fetch the most recent price history to support the price analysis task.
        try:
            stock = yf.Ticker(ticker)
            # Retrieve 1 month of historical data.
            hist = stock.history(period="1mo")
            if hist.empty:
                return f"No price data available for {ticker}."
            # Return the last 3 entries of the historical data for a concise overview.
            return hist.tail(3).to_string()
        except Exception as exc:
            return f"Price data lookup failed: {exc}"


# ---- Agent definition ----
# The project description requires specialized agents working together on the workflow.
# The stock analyst collects evidence, while the report writer turns results into an investor-friendly summary.
stock_analyst = Agent(
    role="Stock Analyst",
    goal="Analyze recent stock data and news",
    backstory="Expert in financial trends, macro indicators, and company performance",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True,
)

report_writer = Agent(
    role="Report Generator",
    goal="Write investor-friendly summaries of stock analysis",
    backstory="Professional writer with expertise in finance reporting",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True,
)

# ---- Task definition ----
# Create tasks for news gathering, trend analysis, and report generation.
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool],
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool],
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer,
)

# ---- Crew assembly ----
# Assemble the crew so the agents can work together in sequence and emit a combined report.
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False,
)

C:\Users\Shubham\AppData\Local\Python\pythoncore-3.13-64\Lib\collections\__init__.py:450: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000002178B9B7010>
  @classmethod
C:\Users\Shubham\AppData\Local\Python\pythoncore-3.13-64\Lib\collections\__init__.py:450: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000002178B9B7C40>
  @classmethod
C:\Users\Shubham\AppData\Local\Python\pythoncore-3.13-64\Lib\collections\__init__.py:450: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x00000217F2A3EE30>
  @classmethod
C:\Users\Shubham\AppData\Local\Python\pythoncore-3.13-64\Lib\collections\__init__.py:450: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000002178B9B7D30>
  @classmethod
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\agent\core.py:357: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.fun

In [4]:
# Run the agent crew and print the final investor report.
# This executes the workflow end-to-end so the AgentOps dashboard can capture the traces.
result = await crew.kickoff_async()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\telemetry\telemetry.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\telemetry\telemetry.py:438: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\telemetry\telemetry.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\telemetry\telemetry.py:438: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_a

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\Shubham\AppData\Local\Temp\ipykernel_17768\3535166057.py:67: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Tool stock_news_searcher executed with result: No news results were found....
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  No news results were found for Apple stock.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1626: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1627: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Task: Write a clean investor report using previous analysis and news insights.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Apple (AAPL) Investor Report**                                                                               │
│                                                                                                                 │
│  **Date:** October 26, 2023                                                                                     │
│                                                                                                                 │
│  **Market Summary**                                                                                             │
│                                                                                                                 │
│  This report on Apple (AAPL) is based on a highly limited dataset, comprising only three trading days in July   │
│  2026 (July 16, 17, and 20). It is important to note that this data is future-dated and insufficient for a      │
│  comprehensive historical analysis or to infer current market conditions. No recent news insights for Apple     │
│  stock were found to supplement this analysis.                                                                  │
│                                                                                                                 │
│  Based purely on the available three days in July 2026:                                                         │
│  *   Apple stock traded within an approximate range of $323.68 to $334.99.                                      │
│  *   The stock initially showed a slight upward movement from July 16 to July 17, with closing prices rising    │
│  from $333.26 to $333.74.                                                                                       │
│  *   On July 20, the stock experienced a notable decline, closing at $326.59, marking the lowest close within   │
│  this brief period. This price drop was accompanied by a significant decrease in trading volume, from over 63   │
│  million shares on the prior two days to approximately 53.4 million shares.                                     │
│  *   No dividends or stock splits were recorded during these three days.                                        │
│                                                                                                                 │
│  The observed decrease in volume coinciding with a price drop on July 20 *could* indicate reduced buying        │
│  interest or increased selling pressure during that specific session, but this is a highly speculative          │
│  observation given the extreme data limitations.                                                                │
│                                                                                                                 │
│  **Investment Outlook**                                                                                         │
│                                                                                                                 │
│  Given the severely limited and future-dated nature of the provided data, it is impossible to formulate a       │
│  reliable or meaningful investment outlook for Apple (AAPL). The three days of trading activity from July 2026  │
│  do not provide a basis for assessing past performance trends, future price movements, or the impact of any     │
│  market-moving events.                                 


📊 Final Stock Analysis Report:

**Apple (AAPL) Investor Report**

**Date:** October 26, 2023

**Market Summary**

This report on Apple (AAPL) is based on a highly limited dataset, comprising only three trading days in July 2026 (July 16, 17, and 20). It is important to note that this data is future-dated and insufficient for a comprehensive historical analysis or to infer current market conditions. No recent news insights for Apple stock were found to supplement this analysis.

Based purely on the available three days in July 2026:
*   Apple stock traded within an approximate range of $323.68 to $334.99.
*   The stock initially showed a slight upward movement from July 16 to July 17, with closing prices rising from $333.26 to $333.74.
*   On July 20, the stock experienced a notable decline, closing at $326.59, marking the lowest close within this brief period. This price drop was accompanied by a significant decrease in trading volume, from over 63 million shares on the prior two days